# Garmin Health Tools Playground

This notebook allows you to experiment with each of the 10 Garmin health query tools.

## Available Tools
1. **query_sleep** - Sleep duration, quality, REM/deep/light sleep
2. **query_heart_rate** - Heart rate trends, resting HR, intraday readings
3. **query_stress** - Stress levels and patterns (0-100 scale)
4. **query_body_battery** - Garmin Body Battery energy levels
5. **query_weight** - Weight tracking over time
6. **query_spo2_respiration** - Blood oxygen and respiration rates
7. **query_activities** - Workout summaries: distance, pace, calories
8. **query_activity_detail** - Per-lap and per-second workout details
9. **query_daily_summary** - Daily steps, calories, activity minutes
10. **query_trends** - Long-term health trends by day/week/month/year

## Setup

Configure your environment and import the tools.

In [9]:
import json
import pandas as pd
import os
from pathlib import Path

# Check for API key - set it here for local testing if needed
# GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

# if not GEMINI_API_KEY:
#     print("⚠️  WARNING: GEMINI_API_KEY not found!")
#     print("   Option 1: Set environment variable:")
#     print("     export GEMINI_API_KEY='your-key-here'")
#     print("   Option 2: Set it in the next cell")
# else:
#     print(f"✓ GEMINI_API_KEY is configured")

# Import the server to access tools
from garmin_mcp.server import TOOLS, handle_call_tool
from garmin_mcp.schema_context import TOOL_SCHEMA_MAP
import asyncio

# Fix for Jupyter: allow nested event loops
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    print("⚠️  nest_asyncio not installed. Installing...")
    import subprocess
    subprocess.check_call(["pip", "install", "nest_asyncio"])
    import nest_asyncio
    nest_asyncio.apply()

print("✓ Imports successful")
print(f"✓ {len(TOOLS)} tools available")

✓ Imports successful
✓ 10 tools available


In [10]:
# Local API Key Configuration (for development/testing only)
# Set your key below if not using environment variable
# DO NOT commit this to version control

LOCAL_API_KEY = None  # Set your key here for local testing: "your-key-here"

if LOCAL_API_KEY:
    os.environ["GEMINI_API_KEY"] = LOCAL_API_KEY
    print("✓ Local API key configured")
elif not os.environ.get("GEMINI_API_KEY"):
    print("⚠️  No API key configured. Edit LOCAL_API_KEY above or set GEMINI_API_KEY env var")

## Helper Functions

These functions make it easy to call tools and inspect results.

In [18]:
def call_tool(tool_name: str, query: str) -> dict:
    """Call a tool with a natural language query.
    
    Args:
        tool_name: Name of the tool (e.g. 'query_sleep')
        query: Natural language question
        
    Returns:
        Dict with 'sql', 'results', 'row_count', 'error'
    """
    # Run the async handler
    result = asyncio.run(handle_call_tool(tool_name, {"query": query}))
    
    # Extract and parse the JSON response
    if result and len(result) > 0:
        text_content = result[0].text
        return json.loads(text_content)
    return {"error": "No response from tool"}


def _format_column_name(col: str) -> str:
    """Format a column name for better readability."""
    # Handle function calls
    if col.startswith('SUM(') or col.startswith('AVG(') or col.startswith('COUNT(') or col.startswith('MAX(') or col.startswith('MIN('):
        func_name = col.split('(')[0]
        return f"[{func_name}]"
    if col.startswith('printf'):
        return "[Result]"
    # Truncate very long names
    if len(col) > 40:
        return col[:37] + "..."
    return col


def show_results(response: dict, limit: int = 10) -> None:
    """Display tool response in a readable format.
    
    Args:
        response: Tool response dict
        limit: Maximum rows to display
    """
    print("\n" + "=" * 80)
    
    if response.get("error"):
        print(f"❌ ERROR: {response['error']}")
        print("=" * 80)
        return
    
    row_count = response.get("row_count", 0)
    results = response.get("results", [])
    
    print(f"✓ Query successful")
    print(f"📊 Results: {row_count} rows\n")
    
    # Show generated SQL for transparency
    print("Generated SQL:")
    print("-" * 80)
    print(response['sql'])
    print("-" * 80)
    print()
    
    # Show results as table
    if results:
        df = pd.DataFrame(results[:limit])
        
        # Format column names for readability
        df.columns = [_format_column_name(col) for col in df.columns]
        
        # Set pandas display options for better output
        with pd.option_context('display.max_columns', None, 'display.max_colwidth', None, 'display.width', None):
            print("Results:")
            print(df.to_string(index=False))
        
        if row_count > limit:
            print(f"\n... and {row_count - limit} more rows")
    else:
        print("No results returned")
    
    print("\n" + "=" * 80)


def list_tools() -> None:
    """Display all available tools with descriptions."""
    print("\n" + "=" * 80)
    print("AVAILABLE TOOLS")
    print("=" * 80)
    for i, tool in enumerate(TOOLS, 1):
        print(f"\n{i}. {tool.name}")
        print(f"   {tool.description}")
    print("\n" + "=" * 80)

print("✓ Helper functions defined")

✓ Helper functions defined


In [13]:
# Debug: Check configuration and test a simple query
print("DEBUG: Configuration Check")
print(f"  API Key Set: {'Yes' if os.environ.get('GEMINI_API_KEY') else 'No'}")
print(f"  Tools Available: {len(TOOLS)}")
print()

# Test with a simple query to see raw response
test_response = call_tool(
    "query_sleep",
    "Show sleep data from last 7 days"
)
print("Raw Response:")
print(json.dumps(test_response, indent=2))

DEBUG: Configuration Check
  API Key Set: Yes
  Tools Available: 10

Raw Response:
{
  "sql": "SELECT *\nFROM sleep\nWHERE day >= strftime('%Y-%m-%d', '2026-03-29', '-7 days')\nORDER BY day DESC",
  "row_count": 6,
  "results": [
    {
      "day": "2026-03-27",
      "start": "2026-03-26 22:59:53.000000",
      "end": "2026-03-27 06:38:53.000000",
      "total_sleep": "07:39:00.000000",
      "deep_sleep": "01:43:00.000000",
      "light_sleep": "04:07:00.000000",
      "rem_sleep": "01:49:00.000000",
      "awake": "00:00:00.000000",
      "avg_spo2": null,
      "avg_rr": 17.0,
      "avg_stress": 11.0,
      "score": 94,
      "qualifier": "EXCELLENT"
    },
    {
      "day": "2026-03-26",
      "start": "2026-03-26 00:32:00.000000",
      "end": "2026-03-26 09:00:00.000000",
      "total_sleep": "08:27:00.000000",
      "deep_sleep": "01:18:00.000000",
      "light_sleep": "06:10:00.000000",
      "rem_sleep": "01:00:00.000000",
      "awake": "00:01:00.000000",
      "avg_spo2":

## Test Each Tool

Run the cells below to test each tool. Modify the queries to experiment!

### 1. Sleep Query Tool

In [19]:
# Test query_sleep
response = call_tool(
    "query_sleep",
    "How much did I sleep last week?"
)
show_results(response)


✓ Query successful
📊 Results: 1 rows

Generated SQL:
--------------------------------------------------------------------------------
SELECT SUM(CAST(SUBSTR(total_sleep,1,2) AS INTEGER) + CAST(SUBSTR(total_sleep,4,2) AS INTEGER)/60.0) FROM sleep WHERE day BETWEEN strftime('%Y-%m-%d', 'now', '-7 days') AND strftime('%Y-%m-%d', 'now', '-1 day')
--------------------------------------------------------------------------------

Results:
    [SUM]
46.166667



In [ ]:
# Try other sleep queries
response = call_tool(
    "query_sleep",
    "What is my average REM sleep this month?"
)
show_results(response)

### 2. Heart Rate Query Tool

In [ ]:
# Test query_heart_rate
response = call_tool(
    "query_heart_rate",
    "What is my resting heart rate trend?"
)
show_results(response)

In [ ]:
# Try other heart rate queries
response = call_tool(
    "query_heart_rate",
    "What is my average RHR this year?"
)
show_results(response)

### 3. Stress Query Tool

In [ ]:
# Test query_stress
response = call_tool(
    "query_stress",
    "What are my most stressful days?"
)
show_results(response)

In [ ]:
# Try other stress queries
response = call_tool(
    "query_stress",
    "How does my stress compare weekday vs weekend?"
)
show_results(response)

### 4. Body Battery Query Tool

In [ ]:
# Test query_body_battery
response = call_tool(
    "query_body_battery",
    "What was my body battery at the start of the day?"
)
show_results(response)

In [ ]:
# Try other body battery queries
response = call_tool(
    "query_body_battery",
    "Body battery trend this week?"
)
show_results(response)

### 5. Weight Query Tool

In [ ]:
# Test query_weight
response = call_tool(
    "query_weight",
    "What is my weight trend this year?"
)
show_results(response)

In [ ]:
# Try other weight queries
response = call_tool(
    "query_weight",
    "How much weight have I lost in the last 3 months?"
)
show_results(response)

### 6. SpO2 and Respiration Query Tool

In [ ]:
# Test query_spo2_respiration
response = call_tool(
    "query_spo2_respiration",
    "What is my average SpO2 during sleep?"
)
show_results(response)

In [ ]:
# Try other SpO2/respiration queries
response = call_tool(
    "query_spo2_respiration",
    "Respiration rate trend this week?"
)
show_results(response)

### 7. Activities Query Tool

In [ ]:
# Test query_activities
response = call_tool(
    "query_activities",
    "How far did I run last week?"
)
show_results(response)

In [ ]:
# Try other activity queries
response = call_tool(
    "query_activities",
    "Average cycling speed this month?"
)
show_results(response)

### 8. Activity Detail Query Tool

In [ ]:
# Test query_activity_detail
response = call_tool(
    "query_activity_detail",
    "What were my lap splits in my last run?"
)
show_results(response)

In [ ]:
# Try other activity detail queries
response = call_tool(
    "query_activity_detail",
    "Show my heart rate during my long run last Saturday"
)
show_results(response)

### 9. Daily Summary Query Tool

In [ ]:
# Test query_daily_summary
response = call_tool(
    "query_daily_summary",
    "Did I hit my step goal this week?"
)
show_results(response)

In [ ]:
# Try other daily summary queries
response = call_tool(
    "query_daily_summary",
    "Average daily steps this month?"
)
show_results(response)

### 10. Trends Query Tool

In [ ]:
# Test query_trends
response = call_tool(
    "query_trends",
    "How has my resting heart rate changed over the past year?"
)
show_results(response)

In [ ]:
# Try other trend queries
response = call_tool(
    "query_trends",
    "Compare my sleep this month vs last month"
)
show_results(response)

## Experiment with Custom Queries

Use the cells below to test your own queries!

In [ ]:
# Custom query - modify tool_name and query as desired
tool_name = "query_sleep"
custom_query = "How much did I sleep last night?"

response = call_tool(tool_name, custom_query)
show_results(response)

In [ ]:
# Another custom query
tool_name = "query_activities"
custom_query = "What activities did I do this week?"

response = call_tool(tool_name, custom_query)
show_results(response)

## Tool Reference

Run the cell below to see all available tools and their descriptions.

In [ ]:
list_tools()

## Debugging Tips

- **Check the generated SQL**: The tool displays the SQL it generated. This helps understand how your query was interpreted.
- **Try different phrasings**: If a query doesn't work as expected, try rephrasing it.
- **Check date ranges**: Make sure you're querying date ranges that have data.
- **Inspect raw response**: Access the full response dict to see all fields including error messages.

Example:
```python
response = call_tool("query_sleep", "How much did I sleep?")
print(json.dumps(response, indent=2))  # See full response
```